# 第一章 -- 认识LLM


In [ ]:
# %%capture
# !pip install transformers>=4.40.1 accelerate>=0.27.2

In [24]:
# 检查当前机器是可以使用 GPU，并且已经安装了正确的 CUDA 版本
!nvidia-smi

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Fri Oct 11 07:56:04 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 4090        On  | 00000000:A1:00.0 Off |                  Off |
| 30%   30C    P8              17W / 450W |   2854MiB / 24564MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## 1.1 Qwen/Qwen2.5-0.5B-Instruct
假定用户已经对机器学习（ML） 和深度学习（DL） 有一定的了解， 第一部就是加载一个模型进行推理预测。可以对模型和 tokenizer 进行分别加载.

In [2]:
import os
os.environ["HF_HOME"] = "/openbayes/home/huggingface"

In [3]:
!echo $HF_HOME

/openbayes/home/huggingface


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer


# 加载模型和 tokenizer （第一次加载的时候会进行下载）
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="mps",
    torch_dtype="auto",
)
# 查看 模型结构是什么？
print(model)

Loading weights: 100%|██████████| 195/195 [00:03<00:00, 57.76it/s]


Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

In [7]:
# 这里有一个 Special token, 打印看一下是什么token呢？
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
# tokenizer.all_special_tokens
tokenizer.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'pad_token': '<|endoftext|>'}

In [21]:
# 每一个模型的 special token 都是不一样的（但是大同小异）
# tokenizer = AutoTokenizer.from_pretrained("/openbayes/input/input0")
# tokenizer.special_tokens_map

## 1.2 基础使用方式
加载模型之后可以怎么使用呢？可以直接使用原始的 model 和 tokenizer 进行推理；

- step1: 加载模型
- step2: 构建 prompt 和 tokenizer
- step3: 推理和解码

刚刚已经加载过模型了，所以直接进行 step2

In [8]:
prompt = "讲一个猫有关的笑话？"
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

#  想一下诗变成什么格式
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

print(text)
print("====" * 10)
print(model_inputs)

<|system|>
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|end|>
<|user|>
讲一个猫有关的笑话？<|end|>
<|assistant|>

{'input_ids': tensor([[32006,   887,   526,   660, 15556, 29892,  2825,   491,   319,  1982,
          5363, 14293, 29889,   887,   526,   263,  8444, 20255, 29889, 32007,
         32010, 29871,   235,   177,   181, 30287, 30502,   234,   143,   174,
         30417, 31057, 30210,   234,   175,   148, 31852, 30882, 32007, 32001]],
       device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}


In [9]:
# step3: 推理和解码
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

为什么猫不喜欢吃苹果？因为它们会变成“苹果”！


## 1.3 使用 transformers 的 pipeline 简化流程

In [10]:
from transformers import pipeline


# step1: 生成 pipeline
# return_full_text=False 表示只返回新生成的文本，不包含输入的prompt
# return_full_text=True 则会返回完整文本，包含输入的prompt和生成的新文本
generator = pipeline(
    "text-generation", 
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,  # 只返回新生成的文本部分
    max_new_tokens=500,
    do_sample=False
)

# step2: 构建 prompt
messages = [
    {"role": "user", "content": "写一个和猫有关的笑话."}
]

# step3，输出并解码
output = generator(messages)
print(output[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, 

有一天，一只猫咪在寻找睡觉的地方。它走到了一个房间里，看着它的床垫，说：“这个床垫太硬了，我不想在这里睡觉。”


另一只猫咪走进，看着它的床垫，说：“这个床垫太软了，我不想在这里睡觉。”


两只猫咪都感到不满，突然一个猫咪说：“我们都不满，但我们这么不满，我们就不睡觉了。”


In [11]:
output

[{'generated_text': '有一天，一只猫咪在寻找睡觉的地方。它走到了一个房间里，看着它的床垫，说：“这个床垫太硬了，我不想在这里睡觉。”\n\n\n另一只猫咪走进，看着它的床垫，说：“这个床垫太软了，我不想在这里睡觉。”\n\n\n两只猫咪都感到不满，突然一个猫咪说：“我们都不满，但我们这么不满，我们就不睡觉了。”'}]